# **SIMD 与 SIMT 混合编程实践：Transpose算子性能优化**

## 概述

本小节介绍**SIMD 与 SIMT 混合编程**模式下的 Transpose 算子开发与优化实现。

Transpose 是离散访存的场景，在上一节中已经介绍了使用 SIMT 快速完成算子功能开发和性能优化。本节再继续围绕 Transpose 算子展开实践：先使用 SIMD 与 SIMT 混合编程的 SIMT 单元实现一个最简单的 Transpose 算子，然后引入 MTE 搬运单元，完成GM和UB间的数据搬运，并利用MTE与SIMT VF间的流水并行，进一步提升性能。

### 学习前置要求

学习本小节前，建议已经具备以下基础：

- 已学习《SIMT编程模型》中核函数、线程索引、内存层级等内容和Gather 算子编程实战。
- 已学习《SIMD编程模型》中核函数、基于指针的C语言编程等内容。
- 已学习《SIMD 与 SIMT 混合编程模型》(03.05.01-03.05.03)，理解核函数与 VF 函数、`asc_vf_call` 调用方式、UB 内存层级。
- 了解基本的 Ascend C 算子开发和执行流程。

### 学习目标

完成本小节后，开发者应能够：

- 掌握混合编程的典型应用场景：由 MTE 搬运单元负责连续数据搬运，SIMT 单元负责分支计算。
- 学会从访存连续性判断 MTE 的适用边界：连续访问的读/写可由 MTE 搬运加速，离散访问（如 Scatter 的写）由 SIMT 完成。
- 掌握基于 tile 划分、UB 中转、UB padding、双缓冲的 Transpose 算子性能优化路径，并能将其迁移到其他离散访存场景。

### 本节内容

- 环境准备
- 基于混合编程的Transpose 算子实现与性能优化
- 小结

## 1. 环境准备

正式开始学习之前，先执行下方脚本检查 CANN Toolkit 是否可用，并把 CANN 环境变量加载到当前 Jupyter 进程，保证后续能够正常导入相关代码并使用 bisheng 编译器完成算子的开发与编译。

本节所有编译、运行和练习修改都在 `Sources/07.06` 目录下进行，`src` 目录仅作为只读的源码仓库存放原始代码。


In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

WORKSPACE = Path("Sources/07.06")
WORKSPACE.mkdir(parents=True, exist_ok=True)

print(f"CANN Toolkit: {cann_home}")
print(f"Workspace: {WORKSPACE.resolve()}")


## 2. Transpose 算子功能介绍
Transpose算子的主要功能是实现二维数据的转置，计算公式如下：

```text
output(col, row) = input(row, col)
```

**本节实现的算子规格**：

| 项 | 取值 |
| --- | --- |
| 输入 `input` | `(1024 , 1024)`，`float` |
| 输出 `output` | `(1024 , 1024)`，`float` |


## 3. Transpose 算子实现

Transpose 属于典型的离散访存场景：输入按行连续读取，但转置后的输出需要跨行写入，写地址不连续。本节围绕 Transpose 算子，沿着一条 **4 级优化路径** 展开实践：先使用 SIMT 直接访问 GM 完成最基础的实现，暴露非连续写带来的性能瓶颈；再逐步引入 MTE 搬运单元、UB 中转、Thread Block 到 tile 的映射调整、UB padding 缓解 bank 冲突，最后通过双缓冲让 MTE 搬运与 SIMT 计算流水并行，逐步把 Transpose 算子的性能调优到接近 GM 带宽上限。

### 3.1 准备源码工作目录

完成环境准备并明确 Transpose 算子规格后，我们开始动手实践。本节按照下表列出的 4 个优化步骤逐步优化 Transpose 算子，每个步骤的实现放在独立目录中：

| 目录 | 核函数 | 优化点 |
| --- | --- | --- |
| `naive` | `transpose_naive_kernel` | SIMT 直接读写 GM，转置写地址跨行不连续 |
| `ub_loop` | `transpose_ub_loop_kernel` | 引入 MTE 搬运 + 32×32 tile + UB 中转，Thread Block 数固定为硬件 vector core 数，核内循环处理多个 tile |
| `ub_pad` | `transpose_ub_pad_kernel` | 输入 tile 改为 32×40 padding 布局，降低 SIMT 转置读 UB 的 bank 冲突 |
| `ub_pad_db` | `transpose_ub_pad_db_kernel` | 双缓冲（ping/pong）使 MTE2 搬入、SIMT VF 转置、MTE3 搬出流水并行 |

上表 4 个版本的完整源码都已存放在 `src/simd_simt_transpose/` 下。执行下面的单元格，把这些源码拷贝到 `Sources/07.06/simd_simt_transpose/` 作为本节课程的工作目录，后续的编译、运行和修改都在 `Sources` 下完成：

In [ ]:
import shutil
from pathlib import Path

SRC_ROOT = Path("src/simd_simt_transpose")            # 只读源码目录
DST_ROOT = Path("Sources/07.06/simd_simt_transpose")  # 工作目录

VERSIONS = ["naive", "ub_loop", "ub_pad", "ub_pad_db"]

# 清理旧的工作目录，保证每次都从 src 拷贝出干净的一份
if DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

for version in VERSIONS:
    dst = DST_ROOT / version
    dst.mkdir(parents=True, exist_ok=True)
    for pattern in ("*.asc", "*.h", "CMakeLists.txt"):
        for f in sorted((SRC_ROOT / version).glob(pattern)):
            shutil.copy2(f, dst / f.name)
    print(f"{version}: {sorted(p.name for p in dst.iterdir())}")


### 3.2 SIMT 直接访问 GM 实现 Transpose

#### 3.2.1 实现思路与代码

首先在混合编程场景使用 SIMT 直接访问 GM，完成最基础的 Transpose 实现：每个线程从 `input` 连续读取对应元素值，计算出转置后的坐标，直接写到 `output` 对应位置。

我们在 07.05 课程中已经得到明确的基本性能优化手段：数据量较大的场景，要限制启动的核数不超过物理核以减少额外头开销。因此本实现直接采用这一优化手段：限制启动核数在物理核以内，每个线程处理多个元素。

该实现由核函数和 SIMT VF 函数两部分组成，先看作为 Device 侧入口的核函数：

```cpp
__global__ __vector__ void transpose_naive_kernel(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height)
{
    asc_init();
    uint32_t total = width * height;
    asc_vf_call<transpose_simt_naive>(dim3(THREAD_COUNT), output, input, width, height, total);
}
```

核函数接收 GM 上的输入输出数据地址，调用 SIMT VF 函数完成转置计算。

SIMT VF 函数负责实际的坐标变换与读写：

```cpp
__simt_vf__ __launch_bounds__(2048) inline void transpose_simt_naive(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t total)
{
    // for循环用于处理多个数据，也可防止越界处理
    for (uint32_t i = blockIdx.x * blockDim.x + threadIdx.x; i < total; i += gridDim.x * blockDim.x) {
        uint32_t row = i / width;
        uint32_t col = i - row * width;
        // 直接写转置后的GM地址，写入方向跨行不连续。
        output[col * height + row] = input[i];
    }
}
```

SIMT 线程按 grid-stride 循环遍历输入矩阵的每个元素：`input[i]` 按行优先连续读取，而转置后写入的目标地址 `output[col * height + row]` 是按列跨行分布的，同一个 Warp 内相邻线程的写地址会落在输出矩阵的不同行，属于非连续写。由于矩阵转置本身计算量很小，这个非连续写会成为该实现的主要瓶颈。

本实现的完整代码保存在 `simd_simt_transpose_naive.asc` 中，执行下面的单元格查看完整源码：

In [ ]:
!cat Sources/07.06/simd_simt_transpose/naive/simd_simt_transpose_naive.asc

#### 3.2.2 CMake 配置

接下来为当前实现编写 CMake 配置文件。

对应的 `CMakeLists.txt` 内容如下，同样已随源码一起拷贝到工作目录：


```cmake
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")

# find_package(ASC) 用于查找和配置 Ascend C 编译工具链
find_package(ASC REQUIRED)
# 指定项目支持的语言包括 ASC 和 CXX，ASC 表示支持使用毕昇编译器对 Ascend C 编程语言进行编译
project(transpose_naive_sample LANGUAGES ASC CXX)

add_executable(demo
    simd_simt_transpose_naive.asc
)

# 通过编译选项设置 NPU 架构
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)
```

**编译选项说明：**

| 选项 | 说明 |
| --- | --- |
| `--npu-arch=dav-3510` | 指定 NPU 架构版本，`dav-` 后为架构号，Ascend 950PR/Ascend 950DT 对应 `dav-3510` |

> **注意**：在 SIMD 与 SIMT 混合编程场景中，虽然使用 SIMT VF 函数完成计算，但不需要添加 `--enable-simt` 编译选项。

#### 3.2.3 编译运行并采集性能

执行以下命令编译并运行当前实现：


In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/07.06/simd_simt_transpose/naive && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能。`msopprof ./demo` 会运行可执行文件并生成 `OPPROF_{timestamp}_...` 性能数据目录，可用于查看算子基础信息、执行耗时、Pipe 利用率和内存访问情况。

In [ ]:
!cd Sources/07.06/simd_simt_transpose/naive/build && msopprof ./demo

SIMT 直接访问 GM 的实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 优化点 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) |
| --- | --- | --- | --- | --- | --- |
| SIMT 直接访问 GM | `transpose_naive_kernel` | 36.38 | 33.17 | 0.00 | 0.00 |

该实现中，输入按行连续读取，但转置写地址跨行不连续，同一 Warp 内相邻线程的写地址分散到输出矩阵的不同行，导致 GM 写效率较低。由于矩阵转置本身计算量很小，这个非连续写会成为该实现的主要瓶颈，后续优化的核心思路是把非连续访问从 GM 转移到 UB，使 GM 读写都变成连续访问。

### 3.3 引入 UB 中转、MTE 搬运

#### 3.3.1 UB中转与MTE搬运的实现思路

上一实现的瓶颈在于转置写地址跨行不连续。本实现参照 07.05 中引入 UB 作为中转的优化思路，将 GM 的非连续访存转移到访问效率更高的 UB 上。另外，由于 32*32 tile 块数据相对连续规整，混合编程场景还可使用 MTE 进行数据搬运，这将进一步提升访存效率。

详细实现思路如下：

先将矩阵划分为 32×32 的 tile，MTE2 把一个 tile 从 GM 连续搬入 UB，SIMT VF 在 UB 内完成转置访问并写入另一块 UB 输出缓冲区，最后 MTE3 把转置后的 tile 按输出矩阵行方向连续搬回 GM。这样一来，非连续访问被转移到了 UB 内部，GM 侧的读和写都变成连续访问。

先看 MTE 搬运的辅助函数，负责把一个 tile 从 GM 搬运到 UB。这里把 UB 侧每行的跨距 `ub_row_bytes` 做成参数，是为了让同一个函数既能搬入本节这种 32×32 连续布局的 UB，也能在后面 3.4 节搬入带 padding 的 UB（32×40 布局），避免为两种布局各写一遍几乎相同的搬运逻辑：

```cpp
// 把一个32×32 tile从GM连续搬入UB，ub_row_bytes由调用方指定UB侧每行跨距，便于非padding/padding场景复用
__aicore__ inline void copy_gm_tile_to_ub(
    __ubuf__ float* in_tile, __gm__ float* input, uint32_t width, uint32_t tiles_x, uint32_t tile_id,
    uint32_t ub_row_bytes)
{
    uint32_t tile_y = tile_id / tiles_x;
    uint32_t tile_x = tile_id - tile_y * tiles_x;
    uint32_t input_offset = tile_y * TILE_DIM * width + tile_x * TILE_DIM;
    // MTE2按二维搬运把一个32×32 tile从GM连续搬入UB。
    asc_copy_gm2ub_align(
        in_tile, input + input_offset, TILE_DIM, TILE_ROW_BYTES, 0, 0, false,
        asc_load_l2_cache_mode::NORMAL_FIRST_VICTIM, width * sizeof(float), ub_row_bytes);
}
```

其中，`asc_copy_gm2ub_align` 的高维切分搬运接口按 `n_burst` 个连续数据块搬运，每块长度为 `len_burst` 字节：这里把 tile 的每一行当作一个数据块，`n_burst=32`，`len_burst=32*sizeof(float)`，源端相邻行之间的地址间隔 `src_stride` 为矩阵的一整行字节数，目的端 UB 中 tile 按 `ub_row_bytes` 跨距存放。本节调用时传入 `ub_row_bytes=TILE_ROW_BYTES`，即 tile 按 32×32 连续存放，因此 32 行数据被连续搬入 UB。

`l2_cache_mode` 参数用于配置数据在 L2 Cache 中的管理策略，需要传入 `asc_load_l2_cache_mode` 枚举类型的值：本例使用 `NORMAL_FIRST_VICTIM`，表示启用 L2 Cache 并把分配的 Cache Line 标记为高替换优先级。把结果从 UB 搬回 GM 的 `asc_copy_ub2gm_align` 同理，对应的枚举类型为 `asc_store_l2_cache_mode`。

每个 Thread Block 启动的线程数为2048，`2048 / (32 * 32) = 2`, 因此整个线程块能一次处理两个 tile，因此在此之上再包一层循环，依次把两个 tile 搬入 UB：

```cpp
// 把本Thread Block分到的2个tile依次从GM搬入UB
__aicore__ inline void copy_gm_2tile_to_ub(
    __ubuf__ float* in_tile, __gm__ float* input, uint32_t width, uint32_t tiles_x, uint32_t tile_base,
    uint32_t total_tiles)
{
    for (uint32_t local_tile = 0; local_tile < TILES_PER_BLOCK; ++local_tile) {
        uint32_t tile_id = tile_base + local_tile;
        if (tile_id < total_tiles) {
            // 非padding场景下，UB中每个输入tile为连续32×32布局。
            copy_gm_tile_to_ub(in_tile + local_tile * TILE_ELEMENTS, input, width, tiles_x, tile_id, TILE_ROW_BYTES);
        }
    }
}
```


SIMT VF 函数在 UB 内完成转置访问。由于数据搬运单独放在 SIMT VF 外部执行，SIMT VF 内只需要完成 UB 上的数据转置计算，计算复杂度比 SIMT 场景的对应实现更简单，寄存器压力也更小，因此可以先让每个线程块启动 2048 个线程，共同完成一组 Tile（2 个 Tile 块），SIMT VF 实现大致如下：

```cpp
// SIMT VF 函数：在UB内完成2个tile的转置访问
__simt_vf__ __launch_bounds__(THREADS_2048) inline void simt_transpose_2tile(
    __ubuf__ float* out_tile, __ubuf__ float* in_tile, uint32_t tile_base, uint32_t total_tiles)
{
    uint32_t local_tile = threadIdx.y >> 5; // 等价于 threadIdx.y / 32
    uint32_t ty = threadIdx.y & (TILE_DIM - 1); // 等价于 threadIdx.y % 32
    uint32_t tx = threadIdx.x;
    uint32_t tile_id = tile_base + local_tile;
    if (tile_id >= total_tiles) {
        return;
    }
    // SIMT VF在UB内按转置方向读取in_tile，并连续写入out_tile。
    out_tile[local_tile * TILE_ELEMENTS + ty * TILE_DIM + tx] =
        in_tile[local_tile * TILE_ELEMENTS + tx * TILE_DIM + ty];
}
```


2048 个线程按 `dim3(32, 64, 1)` 组织：若`threadIdx.y` 属于[0,31]，则处理第一个 tile, 若属于[32, 63]则处理第二个 tile，用`threadIdx.y /32`计算出该线程处理的是哪个tile，用`threadIdx.y % 32` 计算出对应 tile 内的行坐标。每个线程从 `in_tile` 读取转置前的元素，写到 `out_tile` 中转置后的位置；`out_tile` 按输出 tile 的行方向连续排布，方便后续 MTE3 连续搬出。

矩阵按 32×32 划分后，总 tile 数为 `total_tiles = (width/32) × (height/32)`。本实现每个 Thread Block 启动 2048 个线程，可以同时处理两个 tile，因此每个 Thread Block 一轮处理一组（2 个）tile。

#### 3.3.2 每个核处理多个Tile的实现思路

核函数负责调度 MTE 搬运和 SIMT VF 调用。由于 VF 函数属于 PIPE_V 流水，与 MTE2、MTE3 流水并行，因此需要在各个流水间加上同步。

当输入矩阵较大时，为了限制启动的核数不超过物理核，需要设计处理多Tile的逻辑。

明确Thread Block 的数量与 tile 的映射关系：

把 Thread Block 数限制在物理核数内（通过 `get_vector_core_num` 运行时查询），每个 Thread Block 在核内以 `block_num * TILES_PER_BLOCK` 为步长循环处理多组 tile，直到覆盖所有 tile。这样每个核只会被调度一次，不再需要额外的 Thread Block 排队：

```cpp
// 核函数：Thread Block数固定为物理核数，核内循环处理多组tile
__global__ __vector__ void transpose_ub_loop_kernel(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t total_tiles)
{
    asc_init();
    __ubuf__ float in_tile[TILES_PER_BLOCK][TILE_DIM][TILE_DIM];
    __ubuf__ float out_tile[TILES_PER_BLOCK][TILE_DIM][TILE_DIM];
    uint32_t tiles_x = width / TILE_DIM;
    uint32_t loop_step = block_num * TILES_PER_BLOCK;

    // 固定 Thread Block 数为物理核数，核内循环处理多组 tile。
    for (uint32_t tile_base = block_idx * TILES_PER_BLOCK; tile_base < total_tiles; tile_base += loop_step) {
        asc_lock(PIPE_MTE2, SINGLE_BUFFER_MUTEX);
        copy_gm_2tile_to_ub(&in_tile[0][0][0], input, width, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE2, SINGLE_BUFFER_MUTEX);

        // MTE2搬入完成后，SIMT VF读取UB输入buffer。
        asc_lock(PIPE_V, SINGLE_BUFFER_MUTEX);
        asc_vf_call<simt_transpose_2tile>(
            dim3(TILE_DIM, TILE_DIM * TILES_PER_BLOCK, 1), &out_tile[0][0][0], &in_tile[0][0][0], tile_base,
            total_tiles);
        asc_unlock(PIPE_V, SINGLE_BUFFER_MUTEX);

        // SIMT VF写完输出buffer后，MTE3将结果搬回GM。
        asc_lock(PIPE_MTE3, SINGLE_BUFFER_MUTEX);
        copy_ub_2tile_to_gm(output, &out_tile[0][0][0], height, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE3, SINGLE_BUFFER_MUTEX);
    }
}
```

MTE2 搬入、SIMT VF 计算、MTE3 搬出之间存在数据依赖，因此用同一个 `mutex_id` 通过 `asc_lock`/`asc_unlock` 约束三者的执行顺序。

完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.06/simd_simt_transpose/ub_loop/` 目录下，此处不再重复展示全文。执行以下命令查看源码内容：

In [ ]:
!cat Sources/07.06/simd_simt_transpose/ub_loop/simd_simt_transpose_ub_loop.asc

#### 3.3.3 编译运行并采集性能

执行以下命令编译并运行当前实现：

In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/07.06/simd_simt_transpose/ub_loop && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能。

In [ ]:
!cd Sources/07.06/simd_simt_transpose/ub_loop/build && msopprof ./demo

本实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 优化点 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) |
| --- | --- | --- | --- | --- | --- |
| SIMT 直接访问 GM | `transpose_naive_kernel` | 36.38 | 33.17 | 0.00 | 0.00 |
| 引入 UB 中转、MTE 搬运 | `transpose_ub_loop_kernel` | 18.77 | 11.56 | 3.39 | 2.05 |

引入 MTE 搬运和 UB 中转后，GM 侧的读写都变成连续访问，转置计算被限制在 UB 内部完成，Task Duration 相比 SIMT 直接访问 GM 的实现方式大幅下降（36.38μs → 18.77μs），说明 UB 内的转置访问远比直接读写 GM 高效。

不过，`simt_transpose_2tile` 中 SIMT VF 转置读取 `in_tile` 时，同一时刻 32 个线程按列访问一个 32×32 的 UB tile，这些访问地址在 UB 的 bank/subbank 结构上会集中落在少数 bank 上，产生读读 bank 冲突，从而限制了 `aiv_vec_time` 的下降空间。下一步将通过在 UB 中为输入 tile 增加 padding，改变数据的物理布局来消除这种 bank 冲突。

### 3.4 UB padding 缓解 bank 冲突

#### 3.4.1 UB bank 结构与 bank 冲突原理

上一步已经把 Thread Block 数固定为物理核数，消除了调度层面的额外开销，但 `simt_transpose_2tile` 内部的转置访问仍然存在瓶颈：SIMT VF 转置读取 `in_tile` 时，同一时刻 32 个线程按同一列、不同行读取一个 32×32 的 tile，这些访问会集中落到少数相同的 bank/subbank 资源上，产生读读冲突，限制了并行读取的带宽。

以 Ascend 950PR/Ascend 950DT 为例，UB 划分为 16 个 bank，并组织为 8 个 bank group，每个 bank 又划分为 4 个 subbank。SIMT VF 内若同一个 Warp 内多个线程在同一条 UB 访问指令中访问同一个 bank group 的相同编号 subbank，硬件需要排队处理，从而形成 subbank 冲突并增加访问延迟。

<img src="./images/07_06_simd_simt_transpose/bank_structure.png" alt="bank_structure"  width="1500px" >

SIMT VF 内访问 UB 时的 bank 冲突为更细粒度的 subbank 冲突，主要有以下两类：

- **写写冲突**：多个写操作同时访问同一个 bank group 的相同编号 subbank。
- **读读冲突**：多个读操作同时访问同一个 bank group 的相同编号 subbank。

以未做 padding 的 32×32 tile 为例，`in_tile` 在 UB 中按行优先存储，每行 32 个 `float` 共 `TILE_DIM * sizeof(float) = 128` 字节，恰好跨越 4 个 bank。按照 UB 的地址低位交织规则，`in_tile` 的第一行覆盖 bank0～bank3，第二行覆盖 bank4～bank7，第三行覆盖 bank8～bank11，其余行依次类推。SIMT VF 转置时，同一个 Warp 的 32 个线程读取 `in_tile` 的一列元素，由于行跨度固定为 32 个 `float`，这 32 次访问会集中落到两个 bank group 的 subbank0 上，属于读读冲突场景，如下图所示：

<img src="./images/07_06_simd_simt_transpose/case2_bank.png" alt="case2_bank"  width="1500px" >

要缓解这种冲突，需要改变 `in_tile` 的物理行跨度，让同一列的相邻元素在 UB 物理地址上错开分布。纯 SIMT 场景下，对 32×32 的 `float` tile 只需增加一个 subbank 宽度、即 2 列 padding（32×34 布局）即可避免 subbank 冲突；但在 SIMD 与 SIMT 混合编程场景下，`in_tile` 同时是 MTE2 搬运的目的端，MTE 搬运带 padding 的二维数组时还需要考虑 UB 中相邻行的地址跨度对齐。32×34 布局的行跨度为 `34 * sizeof(float) = 136` 字节，不满足 32 字节对齐；因此本实现把 padding 列数设为 8，采用 32×40 布局，行跨度为 `40 * sizeof(float) = 160` 字节、对应 20 个 subbank，既能错开 UB bank 访问，也满足 MTE 搬运对 UB 行跨度的对齐要求。转置访问同一列时，相邻行元素按 20 个 subbank 的跨度错开，32 个线程的访问不再集中到相同 bank group 的同一编号 subbank，从而降低 SIMT VF 转置访问阶段的 subbank 冲突：

<img src="./images/07_06_simd_simt_transpose/case3_bank.png" alt="case3_bank"  width="1500px" >

需要注意的是，32×40 布局是 SIMD 与 SIMT 混合编程场景下结合 MTE 搬运对齐要求后的折中选择。与纯 SIMT 场景增加 2 列 padding 形成 32×34 布局不同，32×40 布局仍可能存在少量 subbank 冲突，但冲突强度已经明显低于未 padding 的 32×32 布局。

#### 3.4.2 实现思路与代码

本实现沿用上一步固定 Thread Block 数、核内循环处理多组 tile 的结构，仅将输入 tile 的 UB 布局从 32×32 改为 32×40（`TILE_PAD_STRIDE = TILE_DIM + TILE_PAD`，`TILE_PAD = 8`）。输出 tile `out_tile` 仍然按 32×32 连续布局存放，因为 SIMT VF 按输出 tile 的行方向连续写入，可通过访存合并降低写入开销，不需要增加 padding。

搬入辅助函数复用 3.3 节的 `copy_gm_tile_to_ub`（带 `ub_row_bytes` 参数），只是调用时把 `ub_row_bytes` 改传为 `TILE_PAD_STRIDE * sizeof(float)`，让 MTE2 按 padded 布局写入 UB：


```cpp
__aicore__ inline void copy_gm_2tile_to_padded_ub(
    __ubuf__ float* in_tile, __gm__ float* input, uint32_t width, uint32_t tiles_x, uint32_t tile_base,
    uint32_t total_tiles)
{
    for (uint32_t local_tile = 0; local_tile < TILES_PER_BLOCK; ++local_tile) {
        uint32_t tile_id = tile_base + local_tile;
        if (tile_id < total_tiles) {
            // padding场景下，输入tile使用32×40布局以降低转置读UB时的bank冲突。
            copy_gm_tile_to_ub(
                in_tile + local_tile * TILE_PAD_ELEMENTS, input, width, tiles_x, tile_id,
                TILE_PAD_STRIDE * sizeof(float));
        }
    }
}
```


与上一步的调用方式相比，唯一的差异是传给 `copy_gm_tile_to_ub` 的最后一个参数 `ub_row_bytes` 从 `TILE_ROW_BYTES`（32×4=128 字节）改为 `TILE_PAD_STRIDE * sizeof(float)`（40×4=160 字节），即每搬入一行后，UB 中下一行的起始地址跳过 padding 部分；`copy_gm_tile_to_ub` 本身的实现完全不用改动，这也是把 `ub_row_bytes` 做成参数的意义所在。

SIMT VF 转置函数按 padded 布局读取 `in_tile`，写入仍是连续的 32×32 `out_tile`：


```cpp

__simt_vf__ __launch_bounds__(THREADS_2048) inline void simt_transpose_2tile_pad(
    __ubuf__ float* out_tile, __ubuf__ float* in_tile, uint32_t tile_base, uint32_t total_tiles)
{
    uint32_t local_tile = threadIdx.y >> 5;
    uint32_t ty = threadIdx.y & (TILE_DIM - 1);
    uint32_t tx = threadIdx.x;
    uint32_t tile_id = tile_base + local_tile;
    if (tile_id >= total_tiles) {
        return;
    }
    // 输入tile带padding，按32×40物理布局读取；输出tile仍按32×32连续布局写入。
    out_tile[local_tile * TILE_ELEMENTS + ty * TILE_DIM + tx] =
        in_tile[local_tile * TILE_PAD_ELEMENTS + tx * TILE_PAD_STRIDE + ty];
}
```


核函数结构与上一步完全一致，只是把 UB 数组声明从 `[TILE_DIM][TILE_DIM]` 改为 `[TILE_DIM][TILE_PAD_STRIDE]`，并调用 padded 版本的搬运函数和 SIMT VF：

```cpp
__global__ __vector__ void transpose_ub_pad_kernel(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t total_tiles)
{
    asc_init();
    __ubuf__ float in_tile[TILES_PER_BLOCK][TILE_DIM][TILE_PAD_STRIDE];
    __ubuf__ float out_tile[TILES_PER_BLOCK][TILE_DIM][TILE_DIM];
    uint32_t tiles_x = width / TILE_DIM;
    uint32_t loop_step = block_num * TILES_PER_BLOCK;

    for (uint32_t tile_base = block_idx * TILES_PER_BLOCK; tile_base < total_tiles; tile_base += loop_step) {
        asc_lock(PIPE_MTE2, SINGLE_BUFFER_MUTEX);
        copy_gm_2tile_to_padded_ub(&in_tile[0][0][0], input, width, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE2, SINGLE_BUFFER_MUTEX);

        asc_lock(PIPE_V, SINGLE_BUFFER_MUTEX);
        asc_vf_call<simt_transpose_2tile_pad>(
            dim3(TILE_DIM, TILE_DIM * TILES_PER_BLOCK, 1), &out_tile[0][0][0], &in_tile[0][0][0], tile_base,
            total_tiles);
        asc_unlock(PIPE_V, SINGLE_BUFFER_MUTEX);

        asc_lock(PIPE_MTE3, SINGLE_BUFFER_MUTEX);
        copy_ub_2tile_to_gm(output, &out_tile[0][0][0], height, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE3, SINGLE_BUFFER_MUTEX);
    }
}
```

完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.06/simd_simt_transpose/ub_pad/` 目录下，此处不再重复展示全文。

In [ ]:
!cat Sources/07.06/simd_simt_transpose/ub_pad/simd_simt_transpose_ub_pad.asc

#### 3.4.3 编译运行并采集性能

完成 CMake 配置后，执行以下命令编译并运行当前实现：

In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/07.06/simd_simt_transpose/ub_pad && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能。

In [ ]:
!cd Sources/07.06/simd_simt_transpose/ub_pad/build && msopprof ./demo

UB padding 缓解 bank 冲突的实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下（单次采集结果会受设备状态影响，实际数值以本机采集为准）：

| 优化点 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) |
| --- | --- | --- | --- | --- | --- |
| SIMT 直接访问 GM | `transpose_naive_kernel` | 36.38 | 33.17 | 0.00 | 0.00 |
| 引入 UB 中转、MTE 搬运 | `transpose_ub_loop_kernel` | 18.77 | 11.56 | 3.39 | 2.05 |
| UB padding 缓解 bank 冲突 | `transpose_ub_pad_kernel` | 11.55 | 4.47 | 3.33 | 2.03 |

将输入 tile 的 UB 布局从 32×32 改为 32×40 后，SIMT VF 转置读取时不再集中命中两个 bank group 的 subbank0，读读冲突强度明显降低，`aiv_vec_time` 相比上一步明显下降（11.56μs → 4.47μs），`aiv_mte2_time`/`aiv_mte3_time` 基本不变（搬运数据量不变，只是目的端跨步略有增加），Task Duration 相比上一步进一步下降（18.77μs → 11.55μs）。

不过，本实现中 MTE2 搬入、SIMT VF 转置、MTE3 搬出这三个步骤仍然共用同一组 `in_tile`/`out_tile` 缓冲区，通过同一个 `mutex_id` 串行执行：当前 tile 的 MTE3 搬出还没完成，下一轮循环的 MTE2 就无法开始搬入，三个 pipe 之间没有重叠，整体耗时约等于三者耗时之和（4.47 + 3.33 + 2.03 ≈ 9.83μs，加上循环调度等开销后接近实测的 11.55μs）。

本实现的仿真指令流水图如下图所示：

<img src="./images/07_06_simd_simt_transpose/case3_trace.png" alt="case3_trace"  width="1500px" >

图中可以看到，`transpose_ub_pad_kernel` 使用单组缓冲区串行处理每组 tile：MTE2 搬入完成后，SIMT VF 才能开始读取输入 buffer；SIMT VF 计算完成后，MTE3 才能搬出输出 buffer；MTE3 搬出结束后，才能复用同一组 UB buffer 进入下一轮 MTE2 搬入。因此 MTE2、SIMT VF、MTE3 三条流水之间存在明显的串行等待，SIMT VF 的执行间隔中会出现较多由搬运和同步带来的空隙。下一步将引入 ping/pong 双缓冲，让 MTE2 搬入下一组 tile 与当前组的 SIMT VF 转置、MTE3 搬出并行执行，进一步压缩整体耗时。

### 3.5 双缓冲实现流水并行

#### 3.5.1 实现思路与代码

上一步（UB padding 缓解 bank 冲突的实现）中 MTE2 搬入、SIMT VF 转置、MTE3 搬出三个步骤共用同一组 UB 缓冲区，通过同一个 `mutex_id` 强制串行：必须等当前 tile 组的 MTE3 搬出完成，才能开始下一组的 MTE2 搬入。本实现引入 **ping/pong 双缓冲**：准备两组独立的 `in_tile`/`out_tile` 缓冲区，让当前循环的 MTE2 搬入使用一组缓冲区时，上一轮的 SIMT VF 转置和 MTE3 搬出可以使用另一组缓冲区并行执行，从而让 `PIPE_MTE2`、`PIPE_V`、`PIPE_MTE3` 三条流水线重叠起来。

双缓冲需要两组独立的 mutex：输入缓冲区的锁和输出缓冲区的锁分别管理，且每组缓冲区各自有一把锁。核函数结构如下：

```cpp
__global__ __vector__ void transpose_ub_pad_db_kernel(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t total_tiles)
{
    asc_init();
    // 两组UB buffer轮换使用：当前buffer进入SIMT VF/MTE3流水，下一组buffer由MTE2搬入。
    __ubuf__ float in_tile[2][TILES_PER_BLOCK][TILE_DIM][TILE_PAD_STRIDE];
    __ubuf__ float out_tile[2][TILES_PER_BLOCK][TILE_DIM][TILE_DIM];
    uint32_t tiles_x = width / TILE_DIM;
    uint32_t loop_step = block_num * TILES_PER_BLOCK;
    uint32_t loop_count = 0;

    for (uint32_t tile_base = block_idx * TILES_PER_BLOCK; tile_base < total_tiles; tile_base += loop_step) {
        uint32_t curr_buffer = loop_count & 1;
        uint8_t input_mutex = DB_INPUT_MUTEX_BASE + static_cast<uint8_t>(curr_buffer);
        uint8_t output_mutex = DB_OUTPUT_MUTEX_BASE + static_cast<uint8_t>(curr_buffer);

        asc_lock(PIPE_MTE2, input_mutex);
        copy_gm_2tile_to_padded_ub(&in_tile[curr_buffer][0][0][0], input, width, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE2, input_mutex);

        // SIMT VF等待当前输入buffer搬入完成；复用输出buffer前等待其上一次MTE3搬出完成。
        asc_lock(PIPE_V, input_mutex);
        asc_lock(PIPE_V, output_mutex);
        asc_vf_call<simt_transpose_2tile_pad>(
            dim3(TILE_DIM, TILE_DIM * TILES_PER_BLOCK, 1), &out_tile[curr_buffer][0][0][0],
            &in_tile[curr_buffer][0][0][0], tile_base, total_tiles);
        asc_unlock(PIPE_V, input_mutex);
        asc_unlock(PIPE_V, output_mutex);

        asc_lock(PIPE_MTE3, output_mutex);
        copy_ub_2tile_to_gm(output, &out_tile[curr_buffer][0][0][0], height, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE3, output_mutex);
        ++loop_count;
    }
}
```


与上一步相比，主要变化是：

- `in_tile`/`out_tile` 各声明 2 组（下标 0、1），`curr_buffer = loop_count & 1` 按循环次数在两组之间轮换。
- 输入缓冲区和输出缓冲区各用一对独立的 `mutex_id`（`DB_INPUT_MUTEX_BASE`/`DB_OUTPUT_MUTEX_BASE` 加上 `curr_buffer` 偏移），而不是像上一步那样所有 pipe 共用一把锁。这样当前轮的 MTE2 只需要等待"同一组"输入缓冲区上一次的读取完成，不需要等待另一组缓冲区的状态，从而让不同组的 MTE2/MTE3 可以与另一组的 SIMT VF 计算重叠执行。
- SIMT VF 转置需要同时持有输入缓冲区和输出缓冲区的锁：读取当前组 `in_tile` 前要等待对应的 MTE2 搬入完成，写入当前组 `out_tile` 前要等待该组上一轮的 MTE3 搬出完成（避免覆盖尚未搬出的数据）。

MTE 搬运辅助函数（`copy_gm_2tile_to_padded_ub`、`copy_ub_2tile_to_gm`）和 SIMT VF 函数（`simt_transpose_2tile_pad`）都直接复用上一步的实现，无需改动。

完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.06/simd_simt_transpose/ub_pad_db/` 目录下，此处不再重复展示全文。

In [ ]:
!cat Sources/07.06/simd_simt_transpose/ub_pad_db/simd_simt_transpose_ub_pad_db.asc

#### 3.5.2 编译运行并采集性能

完成 CMake 配置后，执行以下命令编译并运行当前实现：

In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/07.06/simd_simt_transpose/ub_pad_db && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能。

In [ ]:
!cd Sources/07.06/simd_simt_transpose/ub_pad_db/build && msopprof ./demo

双缓冲流水并行的实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下（单次采集结果会受设备状态影响，实际数值以本机采集为准）：

| 优化点 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) |
| --- | --- | --- | --- | --- | --- |
| SIMT 直接访问 GM | `transpose_naive_kernel` | 36.38 | 33.17 | 0.00 | 0.00 |
| 引入 UB 中转、MTE 搬运 | `transpose_ub_loop_kernel` | 18.77 | 11.56 | 3.39 | 2.05 |
| UB padding 缓解 bank 冲突 | `transpose_ub_pad_kernel` | 11.55 | 4.47 | 3.33 | 2.03 |
| 双缓冲流水并行 | `transpose_ub_pad_db_kernel` | 6.87 | 4.33 | 3.80 | 2.52 |

引入 ping/pong 双缓冲后，`PIPE_MTE2`、`PIPE_V`、`PIPE_MTE3` 三条流水线可以相互重叠：当前组的 SIMT VF 转置和 MTE3 搬出执行时，下一组的 MTE2 搬入已经在另一组缓冲区上开始。Task Duration 相比上一步进一步下降（11.55μs → 6.87μs），且这个值远低于三条 pipe 耗时之和（4.33 + 3.80 + 2.52 ≈ 10.65μs），说明三条流水线确实发生了重叠；但 6.87μs 仍高于三者中最长的 `aiv_vec_time`（4.33μs），说明重叠不是完全理想的（tile 之间的锁同步、循环调度等仍会带来一部分无法隐藏的开销），实测收益介于"完全串行（约 10.65μs）"和"理想全重叠（约 4.33μs）"之间。

本实现的仿真指令流水图如下图所示：

<img src="./images/07_06_simd_simt_transpose/case4_trace.png" alt="case4_trace"  width="1500px" >

对比上一步（单缓冲）与本步（双缓冲）的两张流水图可以看到：单缓冲版本中 MTE2、SIMT VF、MTE3 依次首尾相接，下一轮 MTE2 必须等本轮 MTE3 完全结束才能开始；双缓冲版本中，当前组的 SIMT VF 转置、MTE3 搬出正在执行时，下一组的 MTE2 搬入已经提前在另一组缓冲区上发起，MTE2、SIMT VF、MTE3 在相邻轮次之间形成了重叠。图中仍然保留少量等待间隙，这部分等待用于保护 buffer 复用（同一组缓冲区被再次写入前必须等待对应的读取/搬出完成）以及跨流水的数据依赖，对应前面提到的 6.87μs 与理想全重叠 4.33μs 之间的差距。

至此，Transpose 算子从最初的 SIMT 直接访问 GM（36.38μs），经过 MTE + UB 中转、UB padding 缓解 bank 冲突，最终通过双缓冲实现了流水并行（6.87μs），Task Duration 下降到初始版本的约 19%，完整走完了本节设计的 4 级优化路径。

**与理论带宽上限的对比**：本实现每次搬运读、写各一遍 `1024×1024` 的 `float` 矩阵，总数据量 `D = 1024 × 1024 × 4B × 2 = 8.39MB`。按 Ascend 950PR 理论 GM 峰值带宽 1.6TB/s 计算，理论下限耗时为：

```text
T_theory = 8.39MB / 1.6TB/s ≈ 5.243μs
```

本实现最终 Task Duration 为 6.87μs，对应等效 GM 读写带宽约为：

```text
8.39MB / 6.87μs ≈ 1.22TB/s
```

约达到理论峰值带宽的 `1.22 / 1.6 ≈ 76%`。这个结果已经比较接近理论带宽上限，但仍高于 5.243μs 的理论耗时下限，差距主要来自：UB 内部的读写开销、`PIPE_MTE2`/`PIPE_V`/`PIPE_MTE3` 之间的同步等待、`asc_vf_call` 调用本身的开销、tile 坐标的地址计算，以及双缓冲下流水并未完全重叠（如前面流水图中仍存在的等待间隙）。这些开销大多是当前实现路径下比较固定的成本，如果要进一步逼近理论带宽，需要从减少同步次数、增大单次搬运粒度等方向继续探索，但收益会越来越有限。

## 4. 小结

本节以 Transpose 算子为例，展示了 SIMD 与 SIMT 混合编程中一条典型的性能优化路径：

- **SIMT 直接访问 GM**：实现最简单直观，但转置写地址跨行不连续，GM 写效率低，暴露了离散访存场景下的性能瓶颈。
- **MTE 搬运 + UB 中转**：引入 MTE 搬运单元和 32×32 tile 划分，把转置计算限制在 UB 内部完成：GM 侧的读和写都变成连续访问，SIMT 只负责 UB 内的离散重排。
- **UB padding 缓解 bank 冲突**：针对 SIMT 转置读取 UB 时的 bank/subbank 冲突，把输入 tile 的物理布局由 32×32 改为同时满足 MTE 对齐要求的 32×40 padding，进一步释放了 SIMT 计算侧的并行读取带宽。
- **双缓冲流水并行**：引入 ping/pong 双缓冲，让 MTE2 搬入、SIMT VF 转置、MTE3 搬出三条流水线相互重叠，把原本串行的三段耗时压缩到接近其中最长的一段。

四个优化步骤层层递进，共同印证了 SIMD 与 SIMT 混合编程的适用的典型场景：**MTE 负责连续数据搬运，SIMT 负责局部的离散计算与访存**。Transpose 算子的离散性来自转置这一固定的几何映射，因此可以通过 tile 化和 UB 中转，把原本发生在 GM 上的离散写"转移"到 UB 内部，再借助 MTE 把 GM 侧的读写都还原为连续访问；物理存储布局、流水并行度则是在此基础上进一步压榨性能。

下面的课后练习换一个角度：来看一个输入、输出在 GM 上都能保持连续访问的算子——MaxPool，练习如何在不需要解决离散访问问题的前提下，直接把 MTE + UB 中转 + 双缓冲这套组合技巧用到位。

## 课后编程习题：MaxPool 算子的 MTE 双缓冲优化

**算子语义**：MaxPool（最大池化）把输入矩阵划分成互不重叠的窗口，每个窗口内取最大值作为输出。本习题使用 `4 x 4` 窗口、步长为 `4`（不重叠），计算公式为：

```text
output(i, j) = max(input[4i:4i+4, 4j:4j+4])
```

也就是说，输出矩阵中的每个元素，是输入矩阵中对应 `4 x 4` 窗口内 16 个元素的最大值。由于窗口不重叠，且 `1024 / 4` 能整除，不需要考虑边界 padding。这个算子和上一节（07.05 纯 SIMT）练习使用的是同一个语义和规格，区别在于本节要求你用 **SIMD 与 SIMT 混合编程**的方式实现：MTE 负责 GM/UB 间连续搬运，SIMT VF 负责窗口内的最大值规约。

**规格：**

| 项 | 取值 |
| --- | --- |
| 核函数名 | `maxpool_ub_db_custom` |
| 输入 `input` | `(1024, 1024)`，`float` |
| 输出 `output` | `(256, 256)`，`float` |
| 窗口/步长 | `4 x 4`，不重叠 |

和 Transpose 不同，MaxPool 的输入、输出在 GM 上都是连续访问：把输入按每 `4` 行分成一个 band（band 大小为 `4 x 1024`），一个 band 经过池化后恰好对应输出矩阵的一整行（`256` 个元素）。band 内读取、band 输出写回都不需要跨行跳跃，因此本习题不需要像正文那样先解决非连续访问的问题（3.3~3.4 节），可以直接把 MTE2/MTE3 搬运和双缓冲用在最朴素的实现上。

**要求**：请参照本节 3.5 节 `transpose_ub_pad_db_kernel` 的双缓冲结构（`asc_lock`/`asc_unlock` 管理 `PIPE_MTE2`/`PIPE_V`/`PIPE_MTE3` 三条流水，两组独立的输入/输出缓冲区 + `curr_buffer` 轮换），自己实现混合编程版本的 MaxPool 双缓冲优化：

1. 把输入按每 `4` 行划分成一个 band，共 `256` 个 band；每个 Thread Block 通过 grid-stride 循环处理多个 band（步长为 `block_num * BANDS_PER_BLOCK`，本习题里每个 Thread Block 一次处理 `BANDS_PER_BLOCK = 2` 个 band）。
2. `copy_gm_2band_to_ub`：用 `asc_copy_gm2ub_align` 把本 Thread Block 分到的 band（GM 上是连续的 `4 * 1024` 个 `float`）搬入 UB 输入缓冲区。
3. `maxpool_band_reduce`（SIMT VF 函数）：每个线程负责一个输出列，对其 `4 x 4` 窗口做最大值规约（用三目比较逐元素累积，不要使用 `std::max`/`fmaxf`），写入 UB 输出缓冲区对应位置。
4. `copy_ub_2band_to_gm`：用 `asc_copy_ub2gm_align` 把规约结果（每个 band 对应 `out_width` 个连续 `float`，天然连续）搬回 GM。
5. 核函数中用两组独立的输入/输出 mutex（`DB_INPUT_MUTEX_BASE`/`DB_OUTPUT_MUTEX_BASE` 加上 `curr_buffer` 偏移）管理双缓冲：当前迭代的 MTE2 搬入使用一组缓冲区时，上一轮的 SIMT VF 规约和 MTE3 搬出可以使用另一组缓冲区并行执行。

下面的代码骨架已经搭好双缓冲的循环结构和锁的调用序列，需要你补全三处 `TODO`：`copy_gm_2band_to_ub` 中的搬入参数、`maxpool_band_reduce` 中的窗口规约逻辑、`copy_ub_2band_to_gm` 中的搬出参数。

下面通过 `%%writefile` 把骨架代码写入工作目录 `Sources/07.06/simd_simt_maxpool/`，你直接在 notebook 的 code cell 中补全 `TODO`，再重新执行该 cell 即可覆盖写入，无需打开其他目录的文件。

先创建工作目录：

In [ ]:
!mkdir -p Sources/07.06/simd_simt_maxpool

下面这个 cell 会把骨架代码写入 `Sources/07.06/simd_simt_maxpool/simd_simt_maxpool.asc`。请在本 cell 中补全三处 `TODO`，然后执行本 cell 完成写入（源码同时保留在 `src/simd_simt_maxpool/simd_simt_maxpool.asc`，可作为对照）：

In [ ]:
%%writefile Sources/07.06/simd_simt_maxpool/simd_simt_maxpool.asc
#include <algorithm>
#include <cmath>
#include <iostream>
#include <iterator>
#include <vector>
#include "acl/acl.h"
#include "c_api/asc_simd.h"
#include "simt_api/asc_simt.h"

constexpr uint32_t WINDOW = 4;        // 池化窗口/步长，4x4 不重叠
constexpr uint32_t BAND_ROWS = WINDOW; // 一个 band 含 WINDOW 行输入
constexpr uint32_t BANDS_PER_BLOCK = 2;
constexpr uint8_t DB_INPUT_MUTEX_BASE = 0;
constexpr uint8_t DB_OUTPUT_MUTEX_BASE = 2;

// 把本 Thread Block 分到的 2 个 band 依次从 GM 搬入 UB：每个 band 在 GM 上是 BAND_ROWS*width 个连续 float
__aicore__ inline void copy_gm_2band_to_ub(
    __ubuf__ float* in_band, __gm__ float* input, uint32_t width, uint32_t band_base, uint32_t total_bands)
{
    for (uint32_t local_band = 0; local_band < BANDS_PER_BLOCK; ++local_band) {
        uint32_t band_id = band_base + local_band;
        if (band_id < total_bands) {
            // TODO: 用 asc_copy_gm2ub_align 把 input 中第 band_id 个 band（BAND_ROWS*width 个连续 float）搬到 in_band + local_band * BAND_ROWS * width
        }
    }
}

// 把本 Thread Block 分到的 2 个 band 的池化结果（各 out_width 个 float，天然连续）依次搬回 GM
__aicore__ inline void copy_ub_2band_to_gm(
    __gm__ float* output, __ubuf__ float* out_band, uint32_t out_width, uint32_t band_base, uint32_t total_bands)
{
    for (uint32_t local_band = 0; local_band < BANDS_PER_BLOCK; ++local_band) {
        uint32_t band_id = band_base + local_band;
        if (band_id < total_bands) {
            // TODO: 用 asc_copy_ub2gm_align 把 out_band + local_band * out_width 处的 out_width 个 float 搬到 output 中第 band_id 行输出
        }
    }
}

// SIMT VF 函数：每线程负责一个输出列，对其 4x4 窗口做最大值规约
__simt_vf__ __launch_bounds__(2048) inline void maxpool_band_reduce(
    __ubuf__ float* out_band, __ubuf__ float* in_band, uint32_t width, uint32_t out_width, uint32_t band_base,
    uint32_t total_bands)
{
    uint32_t local_band = threadIdx.y;
    uint32_t out_col = threadIdx.x;
    uint32_t band_id = band_base + local_band;
    if (band_id >= total_bands) {
        return;
    }
    // TODO: 对 in_band 中 local_band 对应 band 里 out_col 列的 4x4 窗口做最大值规约，写入 out_band[local_band * out_width + out_col]
}

// 核函数：ping/pong双缓冲，使MTE2搬入、SIMT VF规约、MTE3搬出流水并行
__global__ __vector__ void maxpool_ub_db_custom(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t out_width,
    uint32_t total_bands)
{
    asc_init();
    // 两组UB buffer轮换使用：当前buffer进入SIMT VF/MTE3流水，下一组buffer由MTE2搬入。
    __ubuf__ float in_band[2][BANDS_PER_BLOCK][BAND_ROWS][1024];
    __ubuf__ float out_band[2][BANDS_PER_BLOCK][256];
    uint32_t loop_step = block_num * BANDS_PER_BLOCK;
    uint32_t loop_count = 0;

    for (uint32_t band_base = block_idx * BANDS_PER_BLOCK; band_base < total_bands; band_base += loop_step) {
        uint32_t curr_buffer = loop_count & 1;
        uint8_t input_mutex = DB_INPUT_MUTEX_BASE + static_cast<uint8_t>(curr_buffer);
        uint8_t output_mutex = DB_OUTPUT_MUTEX_BASE + static_cast<uint8_t>(curr_buffer);

        asc_lock(PIPE_MTE2, input_mutex);
        copy_gm_2band_to_ub(&in_band[curr_buffer][0][0][0], input, width, band_base, total_bands);
        asc_unlock(PIPE_MTE2, input_mutex);

        // SIMT VF等待当前输入buffer搬入完成；复用输出buffer前等待其上一次MTE3搬出完成。
        asc_lock(PIPE_V, input_mutex);
        asc_lock(PIPE_V, output_mutex);
        asc_vf_call<maxpool_band_reduce>(
            dim3(out_width, BANDS_PER_BLOCK, 1), &out_band[curr_buffer][0][0], &in_band[curr_buffer][0][0][0], width,
            out_width, band_base, total_bands);
        asc_unlock(PIPE_V, input_mutex);
        asc_unlock(PIPE_V, output_mutex);

        asc_lock(PIPE_MTE3, output_mutex);
        copy_ub_2band_to_gm(output, &out_band[curr_buffer][0][0], out_width, band_base, total_bands);
        asc_unlock(PIPE_MTE3, output_mutex);
        ++loop_count;
    }
}

// Host 侧 golden 函数：output(i, j) = max(input[4i:4i+4, 4j:4j+4])
void maxpool_golden(
    const std::vector<float>& input, std::vector<float>& golden, uint32_t width, uint32_t height, uint32_t window)
{
    uint32_t out_width = width / window;
    uint32_t out_height = height / window;
    for (uint32_t i = 0; i < out_height; ++i) {
        for (uint32_t j = 0; j < out_width; ++j) {
            float max_val = input[(i * window) * width + j * window];
            for (uint32_t r = 0; r < window; ++r) {
                for (uint32_t c = 0; c < window; ++c) {
                    float v = input[(i * window + r) * width + j * window + c];
                    max_val = (v > max_val) ? v : max_val;
                }
            }
            golden[i * out_width + j] = max_val;
        }
    }
}

// 结果校验函数
uint32_t verify_result(std::vector<float>& output, std::vector<float>& golden)
{
    auto print_tensor = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t max_print_size = 20;
        std::cout << name << ": ";
        std::copy(
            tensor.begin(), tensor.begin() + std::min(tensor.size(), max_print_size),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > max_print_size) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    print_tensor(output, "Output");
    print_tensor(golden, "Golden");
    for (size_t i = 0; i < output.size(); ++i) {
        if (std::fabs(output[i] - golden[i]) > 1e-3f) {
            std::cout << "[Failed] Case accuracy verification failed!" << std::endl;
            return 1;
        }
    }
    std::cout << "[Success] Case accuracy verification passed." << std::endl;
    return 0;
}

// 查询硬件 vector core 数
uint32_t get_vector_core_num(uint32_t device_id)
{
    int64_t core_num = 0;
    aclError ret = aclrtGetDeviceInfo(device_id, ACL_DEV_ATTR_VECTOR_CORE_NUM, &core_num);
    if (ret != ACL_SUCCESS) {
        return 0;
    }
    return static_cast<uint32_t>(core_num);
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t in_height = 1024;
    constexpr uint32_t in_width = 1024;
    constexpr uint32_t in_total_length = in_height * in_width;
    constexpr uint32_t out_width = in_width / WINDOW;
    constexpr uint32_t out_height = in_height / WINDOW;
    constexpr uint32_t out_total_length = out_width * out_height;
    constexpr uint32_t total_bands = in_height / WINDOW;

    // 构造输入数据
    std::vector<float> input(in_total_length);
    for (uint32_t i = 0; i < in_total_length; ++i) {
        input[i] = static_cast<float>((i % 1000) * 1.25f);
    }

    // 计算预期结果
    std::vector<float> golden(out_total_length);
    maxpool_golden(input, golden, in_width, in_height, WINDOW);

    constexpr size_t in_byte_size = in_total_length * sizeof(float);
    constexpr size_t out_byte_size = out_total_length * sizeof(float);

    // 初始化与创建 stream
    aclInit(nullptr);
    int32_t device_id = 0;
    aclrtSetDevice(device_id);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    float* input_device = nullptr;
    float* output_device = nullptr;
    uint8_t* output_host = nullptr;

    // 分配 Host / Device 内存并把数据从 Host 拷贝到 Device
    aclrtMalloc((void**)&input_device, in_byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(input_device, in_byte_size, input.data(), in_byte_size, ACL_MEMCPY_HOST_TO_DEVICE);

    aclrtMalloc((void**)&output_device, out_byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMallocHost((void**)(&output_host), out_byte_size);

    // Thread Block数固定为物理核数
    uint32_t num_blocks = get_vector_core_num(device_id);

    // 启动核函数
    maxpool_ub_db_custom<<<num_blocks, 0, stream>>>(
        output_device, input_device, in_width, in_height, out_width, total_bands);

    // 同步等待核函数执行完成
    aclrtSynchronizeStream(stream);

    // 把结果从 Device 拷回 Host
    aclrtMemcpy(output_host, out_byte_size, output_device, out_byte_size, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> result((float*)output_host, (float*)(output_host + out_byte_size));

    // 释放内存
    aclrtFree(input_device);
    aclrtFree(output_device);
    aclrtFreeHost(output_host);

    // 去初始化
    aclrtDestroyStream(stream);
    aclrtResetDevice(device_id);
    aclFinalize();

    // 校验结果
    return verify_result(result, golden);
}

对应的 `CMakeLists.txt` 同样写入工作目录：

In [ ]:
%%writefile Sources/07.06/simd_simt_maxpool/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")

# find_package(ASC) 用于查找和配置 Ascend C 编译工具链
find_package(ASC REQUIRED)
# 指定项目支持的语言包括 ASC 和 CXX，ASC 表示支持使用毕昇编译器对 Ascend C 编程语言进行编译
project(maxpool_ub_db_sample LANGUAGES ASC CXX)

add_executable(demo
    simd_simt_maxpool.asc
)

# 通过编译选项设置 NPU 架构
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

In [ ]:
!cd Sources/07.06/simd_simt_maxpool && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


请先在上面的 `%%writefile` cell 中补全 TODO 并执行该 cell 完成写入，再执行编译单元格。编译运行成功后，若看到以下输出，则说明你补全的三处 TODO（MTE2 搬入、SIMT VF 窗口规约、MTE3 搬出）逻辑正确：

```text
[Success] Case accuracy verification passed.
```

如果验证失败，可以对照 3.5 节 `transpose_ub_pad_db_kernel` 的双缓冲结构，重点检查两点：`asc_copy_gm2ub_align`/`asc_copy_ub2gm_align` 的搬运参数是否对应到正确的 band 偏移；`maxpool_band_reduce` 里读取的 `in_band` 是否严格对应 `threadIdx.y`（band 内）、`threadIdx.x`（输出列）确定的 `4 x 4` 窗口，而不是搬入整个 band 后随意读取。

**参考答案：**

In [ ]:
!cat answer/07_06_simd_simt_maxpool/simd_simt_maxpool.asc

**小结：Transpose 与 Pool 的访存对比**

回顾正文的 Transpose 优化路径，MTE + UB 中转、UB padding 这两步都是为了解决转置写地址天然不连续的问题；双缓冲则是在此基础上，让 MTE2/SIMT VF/MTE3 三条流水线相互重叠。对比 MaxPool 练习：

- **Transpose**：转置写地址跨行不连续，必须先靠 tile 化 + UB 中转把非连续访问限制在 UB 内部，再用 padding 消除转置读的 bank 冲突，双缓冲才有意义去重叠这三段流水。
- **MaxPool**：窗口不重叠使得输入按 band 读、输出按行写天然连续，从一开始就不存在离散访问问题，因此可以跳过"解决非连续访问"这一整套步骤，直接把 MTE2 搬入、SIMT VF 规约、MTE3 搬出用双缓冲重叠起来。

也就是说，两个算子都用到了同一套 MTE + SIMT VF + 双缓冲的组合技巧，但解决的问题不同：Transpose 是先消除离散访问、再重叠流水线；MaxPool 因为访存天然连续，可以直接进入重叠流水线这一步。如果你也完成了 07.05 节的纯 SIMT 版本练习，可以对比同一个 MaxPool 算子在两种编程模式下的实现差异：纯 SIMT 版本没有 MTE，双缓冲直接作用于 GM 读写；混合编程版本则由 MTE2/MTE3 负责 GM 连续搬运，SIMT VF 只负责 UB 内的窗口规约，双缓冲作用于三条流水线之间的重叠。